# Policy Gradients: REINFORCE from Scratch with NumPy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/policy_gradient_cartpole.ipynb)

This notebook implements the REINFORCE algorithm entirely from scratch in NumPy — forward pass, backpropagation, and RMSProp optimiser, all by hand. No PyTorch, no TensorFlow.

**Blog post:** [Policy Gradients: REINFORCE from Scratch with NumPy](https://sesen.ai/blog/policy-gradients-reinforce-from-scratch)

**Key concepts:**
- Policy gradient theorem: directly optimise action probabilities
- REINFORCE (Williams, 1992): the score function estimator
- Manual backpropagation through a two-layer network
- RMSProp from scratch
- Reward shaping and variance reduction

## Setup

In [ ]:
!pip install -q gymnasium matplotlib

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

## Hyperparameters

These are the original hyperparameters from Berkan's NumPy policy gradient implementation, inspired by [Karpathy's Pong from Pixels](https://karpathy.github.io/2016/05/31/rl/).

In [ ]:
# Original code (CartPole-v0, max 200): H=100, lr=1e-4, gamma=0.95, batch_size=5
# Adapted for CartPole-v1 (max 500): higher gamma and learning rate
H = 100              # hidden layer neurons
batch_size = 5       # episodes per parameter update
learning_rate = 1e-3 # RMSProp learning rate (original: 1e-4, raised for longer episodes)
gamma = 0.99         # discount factor (original: 0.95, raised for longer horizon)
decay_rate = 0.99    # RMSProp decay
epsilon = 1e-5       # RMSProp epsilon

## Network Functions

A two-layer network: 4 (state dim) → 100 (ReLU) → 1 (sigmoid). The output is P(push right).

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def forward(x, model):
    """Forward pass: state → action probability."""
    h = np.dot(model['W1'], x)
    h[h < 0] = 0  # ReLU
    logp = np.dot(model['W2'], h)
    p = sigmoid(logp)
    return p, h

def backward(eph, epdlogp, epx, model):
    """Backward pass: compute weight gradients manually."""
    dW2 = np.dot(eph.T, epdlogp).ravel()
    dh = np.outer(epdlogp, model['W2'])
    dh[eph <= 0] = 0  # backprop through ReLU
    dW1 = np.dot(dh.T, epx)
    return {'W1': dW1, 'W2': dW2}

## Reward Discounting

Standard full-horizon discounting: earlier actions that contributed to a longer run receive more credit.

In [ ]:
def discount_rewards(r, gamma):
    """Standard full-horizon discounting."""
    discounted_r = np.zeros_like(r)
    running_add = 0
    for t in reversed(range(r.size)):
        running_add = running_add * gamma + r[t]
        discounted_r[t] = running_add
    return discounted_r

## Training

The full training loop: collect episodes, compute policy gradients weighted by discounted rewards, update with RMSProp every `batch_size` episodes.

In [ ]:
np.random.seed(42)
env = gym.make('CartPole-v1')
observation, _ = env.reset(seed=42)
D = len(observation)

# Xavier initialisation
model = {
    'W1': np.random.randn(H, D) / np.sqrt(D),
    'W2': np.random.randn(H) / np.sqrt(H),
}
grad_buffer = {k: np.zeros_like(v) for k, v in model.items()}
rmsprop_cache = {k: np.zeros_like(v) for k, v in model.items()}

xs, hs, dlogps, drs, episode_durations = [], [], [], [], []
episode_number = 0
t = 0

while episode_number < 5000:
    x = observation
    aprob, h = forward(x, model)
    action = 1 if np.random.uniform() < aprob else 0

    xs.append(x)
    hs.append(h)
    dlogps.append(action - aprob)  # policy gradient

    observation, reward, terminated, truncated, _ = env.step(action)
    drs.append(reward)

    if terminated or truncated:
        episode_number += 1
        episode_durations.append(t)
        t = 0

        epx = np.vstack(xs)
        eph = np.vstack(hs)
        epdlogp = np.vstack(dlogps)
        epr = np.vstack(drs)
        xs, hs, dlogps, drs = [], [], [], []

        # Discount and standardise rewards
        discounted_epr = discount_rewards(epr, gamma)
        discounted_epr -= np.mean(discounted_epr)
        std = np.std(discounted_epr)
        if std > 0:
            discounted_epr /= std

        # The PG magic: weight gradients by advantage
        epdlogp *= discounted_epr
        grad = backward(eph, epdlogp, epx, model)
        for k in model:
            grad_buffer[k] += grad[k]

        # RMSProp update every batch_size episodes
        if episode_number % batch_size == 0:
            for k, v in model.items():
                g = grad_buffer[k]
                rmsprop_cache[k] = decay_rate * rmsprop_cache[k] + (1 - decay_rate) * g**2
                model[k] += learning_rate * g / (np.sqrt(rmsprop_cache[k]) + epsilon)
                grad_buffer[k] = np.zeros_like(v)

        if episode_number % 500 == 0:
            avg = np.mean(episode_durations[-100:])
            print(f'Episode {episode_number}, 100-ep avg: {avg:.1f}')

        observation, _ = env.reset()
    t += 1

env.close()
print(f'\nFinal 100-episode average: {np.mean(episode_durations[-100:]):.1f}')

## Visualise the Learning

In [ ]:
rolling = np.convolve(episode_durations, np.ones(100)/100, mode='valid')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rolling, 'b-', linewidth=0.8)
ax.axhline(y=500, color='g', linestyle='--', alpha=0.5, label='Max score (500)')
ax.set_xlabel('Episode')
ax.set_ylabel('Duration (100-episode rolling avg)')
ax.set_title('REINFORCE on CartPole-v1')
ax.set_ylim(0, 550)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Understanding the Components

### Forward Pass

The network maps a 4-dimensional state to a single probability:

```
State [x, ẋ, θ, θ̇]  →  Hidden (100 ReLU)  →  Output (sigmoid)  →  P(push right)
```

In [ ]:
# Demonstrate forward pass on a sample state
sample_state = np.array([0.01, -0.02, 0.03, 0.01])  # near-upright pole
prob, hidden = forward(sample_state, model)
print(f'State: {sample_state}')
print(f'P(push right): {prob:.4f}')
print(f'Hidden layer: {hidden.shape[0]} neurons, {np.sum(hidden > 0)} active (ReLU)')

### Reward Discounting Visualisation

Compare what the discounted return looks like across an episode.

In [ ]:
# Simulate a 200-step episode (all +1 rewards)
rewards = np.ones(200)
discounted = discount_rewards(rewards, gamma)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(discounted, 'b-', linewidth=1.5)
axes[0].set_title(f'Discounted returns (gamma={gamma})')
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Return')
axes[0].grid(True, alpha=0.3)

# After standardisation
standardised = discounted.copy()
standardised -= np.mean(standardised)
standardised /= (np.std(standardised) + 1e-8)

axes[1].plot(standardised, 'r-', linewidth=1.5)
axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[1].set_title('After standardisation')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Standardised return')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Raw returns: min={discounted.min():.1f}, max={discounted.max():.1f}')
print(f'Early actions get higher returns → reinforced more strongly')

### Why Standardisation Matters

Without standardisation, all actions in CartPole get positive reinforcement (all rewards are +1). Standardisation centres the returns so ~50% of actions get positive and ~50% get negative signal.

In [ ]:
raw = discount_rewards(rewards, gamma)
standardised = raw.copy()
standardised -= np.mean(standardised)
standardised /= (np.std(standardised) + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(raw, 'r-', linewidth=1.5)
axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[0].set_title('Raw discounted rewards (all positive)')
axes[0].set_xlabel('Timestep')
axes[0].grid(True, alpha=0.3)

axes[1].plot(standardised, 'b-', linewidth=1.5)
axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[1].set_title('After standardisation (zero mean)')
axes[1].set_xlabel('Timestep')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Raw: mean={np.mean(raw):.3f}, std={np.std(raw):.3f}')
print(f'Standardised: mean={np.mean(standardised):.6f}, std={np.std(standardised):.3f}')

## Exercises

### Exercise 1: Remove Reward Shaping

Replace `discount_rewards` with standard full-horizon discounting. How does training speed change?

In [ ]:
# TODO: Retrain the agent using discount_rewards_standard instead of discount_rewards
# Compare the learning curves

### Exercise 2: Vary the Blame Window

Try N ∈ {5, 10, 20, 50}. How does the learning curve respond?

In [ ]:
# TODO: Run training with different N values and plot the learning curves

### Exercise 3: Remove Standardisation

Comment out the mean/std normalisation of rewards. Does the agent still learn?

In [ ]:
# TODO: Retrain without reward standardisation and compare

### Exercise 4: Gamma Sweep

Try γ ∈ {0.8, 0.95, 0.99, 0.999}. How does the discount factor affect learning?

In [ ]:
# TODO: Run training with different gamma values and plot

### Exercise 5: Compare with DQN

Plot the REINFORCE learning curve alongside DQN's on the same axes. Which learns faster? Which is more stable?

In [ ]:
# TODO: Run DQN (from the DQN notebook) and overlay the learning curves